In [ ]:
### (Optional) Run this if you just have this file and nothing else

# 1. Clone the MaxText repository (from AI‑Hypercomputer)
!git clone https://github.com/AI-Hypercomputer/maxtext.git

# 2. Navigate into the cloned directory
%cd maxtext

In [ ]:
### (Optional) Do not run this if you already installed the dependencies

# 3. Ensure setup.sh is executable
!chmod +x setup.sh

# 4. Execute the setup script
!./setup.sh

# force numpy version
!pip install --force-reinstall numpy==2.1.2
#install nest_asyncio
!pip install nest_asyncio

import nest_asyncio
nest_asyncio.apply()
# To fix "This event loop is already running" error in Colab


In [ ]:
# 🚀 Jupyter cell: Download & convert Qwen3-0.6B for MaxText

!pip install -q huggingface_hub git-lfs
!git lfs install

# 1. Download Qwen3-0.6B from Hugging Face
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen3-0.6B", local_dir="/content/Qwen3-0.6B")

# 2. Convert HuggingFace checkpoint → MaxText format
!python /content/maxtext/src/MaxText/convert_qwen3_moe.py \
    --base_model_path /content/Qwen3-0.6B \
    --maxtext_model_path /content/qwen3_06b_maxtext_ckpt \
    --model_size qwen3-0.6b

# 3. Set the checkpoint path for training
MODEL_CHECKPOINT_PATH = "/content/qwen3_06b_maxtext_ckpt"
print("✓ MODEL_CHECKPOINT_PATH set to:", MODEL_CHECKPOINT_PATH)


In [ ]:
import os
import sys
#  Set  home directory. Change this to your home directory where maxtext is cloned
MAXTEXT_HOME = os.path.join("/content", "maxtext")
print(f"Home directory (from Python): {MAXTEXT_HOME}")
#MODEL_CHECKPOINT_PATH = "path/to/scanned/checkpoint"

In [ ]:
from pathlib import Path
from typing import Optional, Dict, Any

# Find MaxText directory and change working directory to it
current_dir = Path.cwd()
if current_dir.name == 'examples':
    # We're in the examples folder, go up one level
    maxtext_path = current_dir.parent.parent
else:
    # We're in the root, MaxText is a subfolder
    maxtext_path = Path(f'{MAXTEXT_HOME}') / 'src' / 'MaxText'

# Change working directory to MaxText project root
os.chdir(maxtext_path)
sys.path.insert(0, str(maxtext_path))

print(f"✓ Changed working directory to: {os.getcwd()}")
print(f"✓ MaxText project root: {maxtext_path}")
print(f"✓ Added to Python path: {maxtext_path}")
import jax
if not jax.distributed.is_initialized():
    jax.distributed.initialize()
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")


In [ ]:
# Hugging Face Authentication Setup
from huggingface_hub import login

# Set your Hugging Face token here
HF_TOKEN = "your_actual_token_here"  # Replace with your actual token
login(token=HF_TOKEN)


In [ ]:
# MaxText imports
try:
    from MaxText import pyconfig
    from MaxText.sft.sft_trainer import train as sft_train

    MAXTEXT_AVAILABLE = True
    print("✓ MaxText imports successful")
except ImportError as e:
    print(f"⚠️ MaxText not available: {e}")
    MAXTEXT_AVAILABLE = False

In [ ]:
# Fixed configuration setup for Qwen-0.6B on small TPU
if MAXTEXT_AVAILABLE:
    config_argv = [
        "",
        f"{MAXTEXT_HOME}/src/MaxText/configs/sft.yml",   # base SFT config
        "model_name=qwen3-0.6b",
        "steps=20",                                     # very short run for testing
        "per_device_batch_size=1",                      # minimal to avoid OOM
        "max_target_length=512",                        # shorter context to fit memory
        "learning_rate=2.0e-5",                         # safe small LR
        "eval_steps=5",
        "weight_dtype=bfloat16",
        "dtype=bfloat16",
        "hf_path=HuggingFaceH4/ultrachat_200k",                       # HuggingFace dataset/model if needed
        f"hf_access_token={HF_TOKEN}",
        "base_output_directory=/tmp/maxtext_qwen06",
        "run_name=sft_qwen0.6b_test",
        "tokenizer_path=Qwen/Qwen3-0.6B",                # Qwen tokenizer
        "eval_interval=10",
        "steps=100",
        "profiler=xplane",
    ]

    # Initialize configuration using MaxText's pyconfig
    config = pyconfig.initialize(config_argv)

    print("✓ Fixed configuration loaded:")
    print(f"  - Model: {config.model_name}")
    print(f"  - Dataset: {config.hf_path}")
    print(f"  - Steps: {config.steps}")
    print(f"  - Use SFT: {config.use_sft}")
    print(f"  - Learning Rate: {config.learning_rate}")
else:
    print("MaxText not available - cannot load configuration")

In [ ]:
#  Execute the training using MaxText SFT trainer's train() function
if MAXTEXT_AVAILABLE:
    print("="*60)
    print("EXECUTING ACTUAL TRAINING")
    print("="*60)


    sft_train(config)
